# Assignment 1: Linear Regression using Deep Neural Network
## Boston Housing Price Prediction

**Objective:** Implement housing price prediction using Deep Neural Networks and experiment with different hyperparameters to find the best performing configuration.

**Dataset:** Boston Housing Price Dataset (or California Housing as alternative)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Create output directory if it doesn't exist
import os
output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory created/verified: {os.path.abspath(output_dir)}")

## 2. Load and Explore Dataset

In [ ]:
# Load Boston Housing from CSV
# IMPORTANT: Update the path to your CSV file location
data = pd.read_csv('boston_housing.csv')  # Update this path if needed!

# Display dataset info
print("Dataset columns:", data.columns.tolist())
print("\nDataset shape:", data.shape)
print("\nFirst few rows:")
display(data.head())

# Separate features and target
# MEDV is the target variable (Median Value of homes)
X = data.drop('MEDV', axis=1).values
y = data['MEDV'].values
feature_names = data.drop('MEDV', axis=1).columns.tolist()

print(f"\nNumber of Features: {X.shape[1]}")
print(f"Number of Samples: {X.shape[0]}")
print(f"Feature Names: {feature_names}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Statistical summary
df.describe()

## 3. Data Visualization

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(y, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Price', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of House Prices', fontsize=14, fontweight='bold')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean: {y.mean():.2f}')
axes[0].legend()

axes[1].boxplot(y, vert=True)
axes[1].set_ylabel('Price', fontsize=12)
axes[1].set_title('Box Plot of House Prices', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Standardize features (important for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling completed!")
print(f"Mean of scaled training features: {X_train_scaled.mean():.4f}")
print(f"Std of scaled training features: {X_train_scaled.std():.4f}")

## 5. Define Hyperparameter Configurations

In [ ]:
# Define simplified hyperparameter configurations
# We'll experiment with:
# 1. Different activation functions (ReLU, Tanh, Sigmoid)
# 2. Different batch sizes (32, 64, 128)

hyperparameter_configs = [
    {
        'name': 'Config 1: ReLU, Batch 32',
        'layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 50,
        'activation': 'relu',
        'dropout': 0.2
    },
    {
        'name': 'Config 2: ReLU, Batch 64',
        'layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 64,
        'epochs': 50,
        'activation': 'relu',
        'dropout': 0.2
    },
    {
        'name': 'Config 3: ReLU, Batch 128',
        'layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 128,
        'epochs': 50,
        'activation': 'relu',
        'dropout': 0.2
    },
    {
        'name': 'Config 4: Tanh, Batch 32',
        'layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 50,
        'activation': 'tanh',
        'dropout': 0.2
    },
    {
        'name': 'Config 5: Tanh, Batch 64',
        'layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 64,
        'epochs': 50,
        'activation': 'tanh',
        'dropout': 0.2
    },
    {
        'name': 'Config 6: Sigmoid, Batch 32',
        'layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 50,
        'activation': 'sigmoid',
        'dropout': 0.2
    },
]

print(f"Total configurations to test: {len(hyperparameter_configs)}")
print("\nWe are experimenting with:")
print("- Activation functions: ReLU, Tanh, Sigmoid")
print("- Batch sizes: 32, 64, 128")
print("- Fixed architecture: [64, 32] (2 hidden layers)")
print("- Learning rate: 0.001 (constant)")
print("- Dropout: 0.2 (constant)")

## 6. Model Building Function

In [ ]:
def create_model(config, input_dim):
    """
    Create a Deep Neural Network model based on configuration.
    
    Args:
        config: Dictionary containing hyperparameters
        input_dim: Number of input features
    
    Returns:
        Compiled Keras model
    """
    model = keras.Sequential()
    
    # Input layer with first hidden layer
    model.add(layers.Dense(
        config['layers'][0],
        activation=config['activation'],
        input_dim=input_dim,
        kernel_initializer='he_normal'
    ))
    
    if config['dropout'] > 0:
        model.add(layers.Dropout(config['dropout']))
    
    # Hidden layers
    for units in config['layers'][1:]:
        model.add(layers.Dense(
            units,
            activation=config['activation'],
            kernel_initializer='he_normal'
        ))
        if config['dropout'] > 0:
            model.add(layers.Dropout(config['dropout']))
    
    # Output layer (linear activation for regression)
    model.add(layers.Dense(1))
    
    # Compile model
    optimizer = keras.optimizers.Adam(learning_rate=config['learning_rate'])
    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )
    
    return model

## 7. Train Models with Different Hyperparameters

In [ ]:
# Store results for comparison
results = []
histories = []
models = []

print("="*80)
print("TRAINING MODELS WITH DIFFERENT HYPERPARAMETER CONFIGURATIONS")
print("="*80)

for i, config in enumerate(hyperparameter_configs):
    print(f"\n{'='*80}")
    print(f"Training: {config['name']}")
    print(f"{'='*80}")
    print(f"Architecture: {config['layers']}")
    print(f"Learning Rate: {config['learning_rate']}")
    print(f"Batch Size: {config['batch_size']}")
    print(f"Activation: {config['activation']}")
    print(f"Dropout: {config['dropout']}")
    
    # Create model
    model = create_model(config, X_train_scaled.shape[1])
    
    # Print model summary for first configuration
    if i == 0:
        print("\nModel Architecture:")
        model.summary()
    
    # Callbacks
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=0
    )
    
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=0
    )
    
    # Train model
    history = model.fit(
        X_train_scaled, y_train,
        batch_size=config['batch_size'],
        epochs=config['epochs'],
        validation_split=0.2,
        callbacks=[early_stopping, reduce_lr],
        verbose=0
    )
    
    # Evaluate on test set
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Store results
    result = {
        'config_name': config['name'],
        'architecture': str(config['layers']),
        'learning_rate': config['learning_rate'],
        'batch_size': config['batch_size'],
        'activation': config['activation'],
        'dropout': config['dropout'],
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'epochs_trained': len(history.history['loss'])
    }
    results.append(result)
    histories.append(history)
    models.append(model)
    
    print(f"\nResults:")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R² Score: {r2:.4f}")
    print(f"  Epochs Trained: {len(history.history['loss'])}")

print(f"\n{'='*80}")
print("ALL MODELS TRAINED SUCCESSFULLY!")
print(f"{'='*80}")

## 8. Results Comparison

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('rmse')
results_df.index = range(1, len(results_df) + 1)

print("\n" + "="*80)
print("PERFORMANCE COMPARISON OF ALL CONFIGURATIONS")
print("="*80)
print(results_df[['config_name', 'rmse', 'mae', 'r2', 'epochs_trained']].to_string())

# Find best model
best_idx = results_df['rmse'].idxmin() - 1  # -1 because index starts from 1
best_config = results[best_idx]

print(f"\n{'='*80}")
print(f"BEST PERFORMING CONFIGURATION")
print(f"{'='*80}")
print(f"Configuration: {best_config['config_name']}")
print(f"Architecture: {best_config['architecture']}")
print(f"Learning Rate: {best_config['learning_rate']}")
print(f"Batch Size: {best_config['batch_size']}")
print(f"Activation: {best_config['activation']}")
print(f"Dropout: {best_config['dropout']}")
print(f"\nPerformance Metrics:")
print(f"  RMSE: {best_config['rmse']:.4f}")
print(f"  MAE:  {best_config['mae']:.4f}")
print(f"  R² Score: {best_config['r2']:.4f}")

## 9. Visualizations

In [ ]:
# Performance metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['rmse', 'mae', 'r2', 'epochs_trained']
titles = ['RMSE Comparison', 'MAE Comparison', 'R² Score Comparison', 'Training Epochs']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    ax = axes[idx // 2, idx % 2]
    
    # Sort by metric for better visualization
    sorted_df = results_df.sort_values(metric, ascending=(metric != 'r2'))
    
    bars = ax.barh(range(len(sorted_df)), sorted_df[metric], color=color, alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels([name.replace('Config ', 'C') for name in sorted_df['config_name']], fontsize=9)
    ax.set_xlabel(metric.upper(), fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels on bars
    for i, (bar, value) in enumerate(zip(bars, sorted_df[metric])):
        ax.text(value, i, f' {value:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/assignment1_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Training history - Loss curves
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for idx, (history, config) in enumerate(zip(histories, hyperparameter_configs)):
    if idx < len(axes):
        ax = axes[idx]
        
        epochs = range(1, len(history.history['loss']) + 1)
        
        ax.plot(epochs, history.history['loss'], 'b-', linewidth=2, label='Training Loss', alpha=0.8)
        ax.plot(epochs, history.history['val_loss'], 'r-', linewidth=2, label='Validation Loss', alpha=0.8)
        
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel('Loss (MSE)', fontsize=10)
        ax.set_title(config['name'].replace('Config ', 'C'), fontsize=11, fontweight='bold')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)

# Remove extra subplots if any
for idx in range(len(histories), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('outputs/assignment1_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Predicted vs Actual for Best Model
best_model = models[best_idx]
y_pred_best = best_model.predict(X_test_scaled, verbose=0).flatten()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(y_test, y_pred_best, alpha=0.5, s=50, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=3, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted Price', fontsize=12, fontweight='bold')
axes[0].set_title(f'Predicted vs Actual - {best_config["config_name"]}', 
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.5, s=50, edgecolors='black', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=3)
axes[1].set_xlabel('Predicted Price', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Residuals', fontsize=12, fontweight='bold')
axes[1].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/assignment1_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Error distribution for best model
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_xlabel('Residuals', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Residuals', fontsize=14, fontweight='bold')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Q-Q plot for normality check
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/assignment1_residuals.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Analysis and Justification

In [ ]:
print("="*80)
print("ANALYSIS OF HYPERPARAMETER EXPERIMENTS")
print("="*80)

print("\n1. BEST PERFORMING CONFIGURATION:")
print(f"   {best_config['config_name']}")
print(f"   RMSE: {best_config['rmse']:.4f}, R²: {best_config['r2']:.4f}")

print("\n2. KEY OBSERVATIONS:")

# Network depth analysis
shallow = results_df[results_df['config_name'].str.contains('Shallow')]
deep = results_df[results_df['config_name'].str.contains('Deep')]
print(f"\n   a) Network Depth:")
if not shallow.empty and not deep.empty:
    print(f"      - Shallow Network RMSE: {shallow['rmse'].values[0]:.4f}")
    print(f"      - Deep Network RMSE: {deep['rmse'].values[0]:.4f}")
    if shallow['rmse'].values[0] < deep['rmse'].values[0]:
        print("      - Shallow network performed better (less overfitting)")
    else:
        print("      - Deep network performed better (better feature extraction)")

# Learning rate analysis
high_lr = results_df[results_df['config_name'].str.contains('Higher Learning')]
print(f"\n   b) Learning Rate:")
if not high_lr.empty:
    print(f"      - Standard LR (0.001) avg RMSE: {results_df[results_df['learning_rate']==0.001]['rmse'].mean():.4f}")
    print(f"      - Higher LR (0.01) RMSE: {high_lr['rmse'].values[0]:.4f}")

# Dropout analysis
with_dropout = results_df[results_df['dropout'] > 0]
without_dropout = results_df[results_df['dropout'] == 0]
print(f"\n   c) Regularization (Dropout):")
if not with_dropout.empty:
    print(f"      - With Dropout avg RMSE: {with_dropout['rmse'].mean():.4f}")
    print(f"      - Without Dropout avg RMSE: {without_dropout['rmse'].mean():.4f}")

# Activation function
relu_models = results_df[results_df['activation'] == 'relu']
tanh_models = results_df[results_df['activation'] == 'tanh']
print(f"\n   d) Activation Function:")
if not relu_models.empty:
    print(f"      - ReLU avg RMSE: {relu_models['rmse'].mean():.4f}")
if not tanh_models.empty:
    print(f"      - Tanh avg RMSE: {tanh_models['rmse'].mean():.4f}")

print("\n3. CONCLUSION:")
print(f"   The best configuration achieves an R² score of {best_config['r2']:.4f},")
print(f"   explaining {best_config['r2']*100:.2f}% of the variance in house prices.")
print(f"   The average prediction error (MAE) is ${best_config['mae']:.2f}.")

## 11. Save Results

In [ ]:
# Save results to CSV
results_df.to_csv('outputs/assignment1_results.csv', index=False)
print("Results saved to 'assignment1_results.csv'")

# Save best model
best_model.save('outputs/assignment1_best_model.keras')
print("Best model saved to 'assignment1_best_model.keras'")